<a href="https://colab.research.google.com/github/youreyesarecameras-dot/WP/blob/main/Weapon_Fight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pygame
import math
import random

# Initialize Pygame
pygame.init()

# --- CONFIGURATION (EDIT THESE TO CHANGE THE GAME!) ---

# Modes: "FFA", "2_TEAMS", "3_TEAMS", "GIGA", "1v1"
CURRENT_MODE = "FFA"
# Sizes: "SMALL" (600x600), "MEDIUM" (800x800), "LARGE" (1000x1000)
MAP_SIZE = "MEDIUM"

# ------------------------------------------------------

# Map Size Definitions
SIZES = {
    "SMALL": (600, 600),
    "MEDIUM": (800, 800),
    "LARGE": (1000, 1000)
}
WIDTH, HEIGHT = SIZES[MAP_SIZE]
FPS = 60

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
TEAM_COLORS = {
    1: (0, 100, 255),    # Blue
    2: (255, 140, 0),    # Orange
    3: (0, 200, 0),      # Green
    "GIGA": (200, 0, 50) # Reddish
}

def get_random_color(exclude_colors):
    while True:
        color = (random.randint(50, 255), random.randint(50, 255), random.randint(50, 255))
        # Simple check to avoid dark colors or duplicate colors
        if sum(color) > 200 and color not in exclude_colors:
            return color

class Weapon:
    def __init__(self, w_type):
        self.type = w_type
        if self.type == "Sword":
            self.dmg = 1.0
            self.length = 35.0
        elif self.type == "Spear":
            self.dmg = 1.0
            self.length = 50.0

    def upgrade(self):
        if self.type == "Sword":
            self.dmg += 1.0
        elif self.type == "Spear":
            self.dmg += 0.2
            self.length += 0.2

class Ball:
    def __init__(self, x, y, team, color, is_giga=False, weapon_type="Sword"):
        self.x = x
        self.y = y
        self.team = team
        self.color = color
        self.is_giga = is_giga

        self.radius = 40 if is_giga else 20
        self.max_hp = 1000 if is_giga else 100
        self.hp = self.max_hp

        self.speed = 1.5 if is_giga else 3.0
        self.angle = random.uniform(0, math.pi * 2)
        self.rotation_speed = 0.05 if is_giga else 0.1

        self.weapon = Weapon(weapon_type)
        self.hit_cooldown = 0 # Prevents taking 60 damage per second from the same weapon
        self.alive = True

    def update(self, enemies):
        if not self.alive: return

        if self.hit_cooldown > 0:
            self.hit_cooldown -= 1

        # Simple AI: Find closest enemy and move towards them
        target = None
        min_dist = float('inf')
        for enemy in enemies:
            if enemy.alive and enemy.team != self.team:
                dist = math.hypot(enemy.x - self.x, enemy.y - self.y)
                if dist < min_dist:
                    min_dist = dist
                    target = enemy

        if target:
            # Move towards target
            dx = target.x - self.x
            dy = target.y - self.y
            dist = math.hypot(dx, dy)
            if dist > 0:
                self.x += (dx / dist) * self.speed
                self.y += (dy / dist) * self.speed

            # Rotate weapon continuously
            self.angle += self.rotation_speed

        # Keep inside bounds
        self.x = max(self.radius, min(WIDTH - self.radius, self.x))
        self.y = max(self.radius, min(HEIGHT - self.radius, self.y))

    def check_attack(self, enemies):
        if not self.alive: return

        # Calculate weapon tip
        tip_x = self.x + math.cos(self.angle) * (self.radius + self.weapon.length)
        tip_y = self.y + math.sin(self.angle) * (self.radius + self.weapon.length)

        for enemy in enemies:
            if enemy.alive and enemy.team != self.team and enemy.hit_cooldown == 0:
                # Check if weapon tip is inside enemy radius
                dist_to_tip = math.hypot(tip_x - enemy.x, tip_y - enemy.y)
                if dist_to_tip <= enemy.radius:
                    enemy.hp -= self.weapon.dmg
                    enemy.hit_cooldown = 15 # Quarter second invulnerability
                    self.weapon.upgrade() # Upgrade weapon on successful hit

                    if enemy.hp <= 0:
                        enemy.alive = False

    def draw(self, screen):
        if not self.alive: return

        # Draw Ball
        pygame.draw.circle(screen, self.color, (int(self.x), int(self.y)), self.radius)
        pygame.draw.circle(screen, BLACK, (int(self.x), int(self.y)), self.radius, 2) # Outline

        # Draw Weapon
        base_x = self.x + math.cos(self.angle) * self.radius
        base_y = self.y + math.sin(self.angle) * self.radius
        tip_x = self.x + math.cos(self.angle) * (self.radius + self.weapon.length)
        tip_y = self.y + math.sin(self.angle) * (self.radius + self.weapon.length)

        # Weapon color based on type
        w_color = (200, 200, 200) if self.weapon.type == "Sword" else (139, 69, 19)
        w_thickness = 4 if self.weapon.type == "Sword" else 2
        pygame.draw.line(screen, w_color, (base_x, base_y), (tip_x, tip_y), w_thickness)

        # Draw HP Bar
        hp_ratio = self.hp / self.max_hp
        bar_width = self.radius * 2
        pygame.draw.rect(screen, (255, 0, 0), (self.x - self.radius, self.y - self.radius - 10, bar_width, 5))
        pygame.draw.rect(screen, (0, 255, 0), (self.x - self.radius, self.y - self.radius - 10, bar_width * hp_ratio, 5))

def setup_game():
    balls = []
    used_colors = []

    if CURRENT_MODE == "FFA":
        num_balls = random.randint(4, 8)
        for i in range(num_balls):
            color = get_random_color(used_colors)
            used_colors.append(color)
            w_type = random.choice(["Sword", "Spear"])
            balls.append(Ball(random.randint(50, WIDTH-50), random.randint(50, HEIGHT-50), team=i, color=color, weapon_type=w_type))

    elif CURRENT_MODE == "2_TEAMS":
        for team in [1, 2]:
            for _ in range(4): # Up to 4 per team
                w_type = random.choice(["Sword", "Spear"])
                x = random.randint(50, WIDTH//2) if team == 1 else random.randint(WIDTH//2, WIDTH-50)
                y = random.randint(50, HEIGHT-50)
                balls.append(Ball(x, y, team=team, color=TEAM_COLORS[team], weapon_type=w_type))

    elif CURRENT_MODE == "3_TEAMS":
        for team in [1, 2, 3]:
            for _ in range(3): # Up to 3 per team
                w_type = random.choice(["Sword", "Spear"])
                balls.append(Ball(random.randint(50, WIDTH-50), random.randint(50, HEIGHT-50), team=team, color=TEAM_COLORS[team], weapon_type=w_type))

    elif CURRENT_MODE == "GIGA":
        # Add Giga Ball
        balls.append(Ball(WIDTH//2, HEIGHT//2, team="GIGA", color=TEAM_COLORS["GIGA"], is_giga=True, weapon_type="Sword"))
        # Add 2-6 challengers
        num_challengers = random.randint(2, 6)
        for i in range(num_challengers):
            color = get_random_color(used_colors)
            used_colors.append(color)
            w_type = random.choice(["Sword", "Spear"])
            balls.append(Ball(random.randint(50, WIDTH-50), random.randint(50, HEIGHT-50), team="CHALLENGERS", color=color, weapon_type=w_type))

    elif CURRENT_MODE == "1v1":
        for i in range(2):
            color = get_random_color(used_colors)
            used_colors.append(color)
            w_type = random.choice(["Sword", "Spear"])
            balls.append(Ball(random.randint(50, WIDTH-50), random.randint(50, HEIGHT-50), team=i, color=color, weapon_type=w_type))

    return balls

def main():
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption(f"Earclacks Clone - Mode: {CURRENT_MODE}")
    clock = pygame.time.Clock()

    balls = setup_game()
    running = True

    while running:
        screen.fill((30, 30, 30)) # Dark gray background

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        # Update Logic
        for ball in balls:
            ball.update(balls)

        # Attack Logic (separated to ensure all move before resolving hits)
        for ball in balls:
            ball.check_attack(balls)

        # Draw Logic
        for ball in balls:
            ball.draw(screen)

        pygame.display.flip()
        clock.tick(FPS)

    pygame.quit()

if __name__ == "__main__":
    main()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


KeyboardInterrupt: 